In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BIBLE_DIR = Path('bible')  # adjust if needed
print(f"Bible dir exists: {BIBLE_DIR.exists()}")

In [ ]:
import pandas as pd
strongs_df = pd.read_csv("greek-enh-lxx-apo-OT-blb.csv", dtype={'strong_num': str})
abp_strongs_df=pd.read_csv("biblehub_LXX_EXT_el_en_direct.csv", dtype={'strong_num': str})

In [ ]:
words_df = strongs_df["form"].apply(lambda f: len(f.split(' ')) if ' ' in f else 1).sum()
words_df

In [ ]:
# ── Load all chapters ──────────────────────────────────────────────────────────
records = []

for book_dir in sorted(BIBLE_DIR.iterdir()):
    if not book_dir.is_dir():# or not book_dir.name=='1_samuel':
        continue
    book = book_dir.name
    for json_file in sorted(
        [f for f in book_dir.glob('*.json') if f.stem.isdigit()], #and f.stem=='20'],
        key=lambda x: int(x.stem)
    ):
        chapter = int(json_file.stem)
        try:
            verses = json.loads(json_file.read_text(encoding='utf-8'))
        except Exception as e:
            print(f"  skip {json_file}: {e}")
            continue

        total_hb_words = 0
        total_has_a_greek_mapped = 0
        total_has_a_latvian_mapped = 0
        total_greek_words = 0
        total_latvian_words = 0
        leftover_greek_words = 0
        leftover_latvian_words = 0

        for verse in verses:
            for hw in verse.get('hebrew_words', []):
                total_hb_words += 1
                if hw.get('greek'):  
                    total_has_a_greek_mapped += 1
                    total_greek_words += len(hw.get('greek'))
                if hw.get('latvian'): 
                    total_has_a_latvian_mapped += 1
                    total_latvian_words += len(hw.get('latvian'))
            leftover_greek_words  += len(verse.get('leftover_greek',  []))
            total_greek_words  += len(verse.get('leftover_greek',  []))
            leftover_latvian_words += len(verse.get('leftover_latvian', []))
            total_latvian_words += len(verse.get('leftover_latvian', []))
        #print(f'{leftover_latvian_words}')
        records.append({
            'book': book,
            'chapter': chapter,
            'total_hb_words': total_hb_words,
            'has_a_greek_mapped': total_has_a_greek_mapped,
            'has_a_latvian_mapped': total_has_a_latvian_mapped,
            'leftover_greek': leftover_greek_words,
            'total_greek_words': total_greek_words,
            'leftover_latvian': leftover_latvian_words,
            'total_latvian_words': total_latvian_words
        })

df = pd.DataFrame(records)
df['greek_usage_pct'] = (df['total_greek_words'] - df['leftover_greek'])/df['total_greek_words']*100
df['latvian_usage_pct'] = (df['total_latvian_words'] - df['leftover_latvian'])/df['total_latvian_words']*100
df['greek_pct']   = df['has_a_greek_mapped']   / df['total_hb_words'].replace(0, pd.NA) * 100
df['latvian_pct'] = df['has_a_latvian_mapped'] / df['total_hb_words'].replace(0, pd.NA) * 100
df['has_leftover_greek']   = df['leftover_greek']   > 0
df['has_leftover_latvian'] = df['leftover_latvian'] > 0
df['has_both_leftovers']   = df['has_leftover_greek'] & df['has_leftover_latvian']

print(f"Loaded {len(df)} chapters across {df['book'].nunique()} books")
df.head()

In [ ]:
t_h=df['total_hb_words'].sum()
t_g=df['total_greek_words'].sum()
t_l=df['total_latvian_words'].sum()
l_g=df['leftover_greek'].sum()
l_l=df['leftover_latvian'].sum()
a_g=df['greek_usage_pct'].mean()#.avg()
a_l=df['latvian_usage_pct'].mean()#.avg()
mn_g=df['greek_usage_pct'].min()#.avg()
mn_l=df['latvian_usage_pct'].min()#.avg()
mx_g=df['greek_usage_pct'].max()#.avg()
mx_l=df['latvian_usage_pct'].max()#.avg()
print(f" stats:\n\
    h: {t_h}\n\
    g: {t_g}\n\
    l: {t_l}\n\
    ")
print(f" leftovers:\n\
    g: {l_g}\n\
    l: {l_l}\n\
    ")
print(f" usages:\n\
    g: avg: {a_g:.2f} min: {mn_g:.2f} max: {mx_g:.2f}\n\
    l: avg: {a_l:.2f} min: {mn_l:.2f} max: {mx_l:.2f}\n\
    ")


In [ ]:
print(words_df)